# 05. Empirical rate exposure: is it real, and how does it relate to refinancing?

**The question.** How large is MBB's empirical duration each month, how precisely is it measured, and does it vary by more than measurement noise?

**Why a portfolio manager cares.** The size of the hedge is the duration estimate. If exposure barely moves, a static hedge is enough and re-hedging effort is wasted. If it moves a lot, the hedge must be updated, and the question becomes how.

**Learning objectives**

1. Estimate one month's duration by hand and confirm the library function.
2. Build the monthly panel of realized durations, standard errors, updating estimates, and the refinancing state for every month (saved for later notebooks).
3. Test variation beyond noise on development months only.
4. Describe the development-period relation between duration and the share in the money.

| | |
|---|---|
| Inputs | `data/processed/daily_panel.parquet` |
| Outputs | `data/processed/monthly_panel.parquet`, `results/exposure/*`, figures `05_*` |
| Run after | 02 |
| Approximate runtime | under a minute |

> **The test period stays closed in this notebook.** The monthly panel is computed for every month because later notebooks need it, but every table and figure shown here uses development months only (through December 2014). The full-sample measurement appears in notebook 08, after the freeze.

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import exposure as ex
from mbshedge import pipeline as pl
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
win = pl.windows(daily, EXP)
dev_last = win["development_end"]
xdir = paths.results / "exposure"; xdir.mkdir(parents=True, exist_ok=True)
print("development months:", win["first_month"], "to", dev_last)

## 1. One month by hand

For month $m$ we regress daily MBS excess returns on same-day and previous-day changes in the 10-year yield:

$$
r_t = a_m + b_{0,m}\,\Delta y_t + b_{1,m}\,\Delta y_{t-1} + e_t , \qquad D_m = -(b_{0,m} + b_{1,m}).
$$

With returns and yield changes both in decimals, $D_m$ is in years. The previous-day term catches a response that arrives a day late, for example when the ETF's price is slow to reflect its holdings. The timing of the two closes works the other way: the 10-year yield is fixed around 3:30 pm and the ETF closes at 4 pm, so a rate move in that half hour enters today's ETF return but tomorrow's yield change. Section 5 measures that effect on development data. The standard error uses Newey–West with 2 lags, which allows for some serial correlation in the residuals, with the HC3 leverage adjustment: days with large yield moves pin down the slope, so their residuals are artificially small, and notebook 04 shows that without the adjustment the error bars are too narrow.

In [ ]:
m = pd.Period("2013-06", "M")                          # the taper tantrum, a development month
g = daily.loc[str(m), ["r_mbb_ex", "dy10"]].copy()
g["dy_lag"] = daily["dy10"].shift(1).loc[str(m)]
X = np.column_stack([np.ones(len(g)), g["dy10"], g["dy_lag"]]); y = g["r_mbb_ex"].to_numpy()
beta = np.linalg.lstsq(X, y, rcond=None)[0]
print(f"{m}: {len(g)} trading days, b0 = {beta[1]:.3f}, b1 = {beta[2]:.3f}, D = -(b0 + b1) = {-(beta[1] + beta[2]):.3f} years")
fit = ex.ols_hac(y, X, lags=EXP["exposure"]["newey_west_lags"], leverage=True)
d_lib, se_lib = ex.duration_from_fit(fit)
plain = ex.duration_from_fit(ex.ols_hac(y, X, lags=EXP["exposure"]["newey_west_lags"]))[1]
print(f"library: D = {d_lib:.3f} years, se = {se_lib:.3f} years (Newey-West with HC3 leverage adjustment; plain Newey-West {plain:.3f}), R^2 = {fit['r2']:.2f}")
assert abs(d_lib + beta[1] + beta[2]) < 1e-10

## 2. The monthly panel for all months (saved, shown for development only)

For each month-end decision date the panel stores: the realized duration $D_m$ and its standard error, the updating (EWMA) duration for every candidate half-life, the information-weighted average of the share in the money $\bar g$ that the hybrid needs, and the refinancing state at the close.

In [ ]:
monthly = pl.monthly_panel(daily, EXP, EXP["exposure"]["ewma_halflife_grid_days"])
monthly.to_parquet(paths.processed / "monthly_panel.parquet")
devm = monthly.loc[win["first_month"]:dev_last]
print(f"Saved {len(monthly)} months. Development months: {len(devm)}, valid: {int(devm['valid'].sum())}")
flagged = devm[~devm["valid"]][["n_days", "D", "flag"]]
print("Development months without a usable estimate:"); display(flagged)

## 3. Q1 on development months: variation beyond noise

Treat each month as a noisy measurement: $D_m = \mu + u_m + \varepsilon_m$, where $\text{Var}(\varepsilon_m) = s_m^2$ is measurement noise and $\text{Var}(u_m) = \tau^2$ is genuine variation in exposure. Cochran's Q tests $\tau = 0$.

Notebook 04 showed that the textbook $\chi^2$ p-value of Q rejects far too often when the monthly error bars come from about 21 days of data. The p-value used here comes from the **constant-duration residual bootstrap**, which re-estimates every month under the null with the same estimator. The size of the variation is reported as a robust $\tau$: the observed variance of the monthly estimates minus the variance they would have from noise alone under the null,

$$
\hat\tau^2_{\text{robust}} = \text{Var}(\hat D_m) - \overline{\text{Var}(\hat D^{*}_m)}_{\text{null}} .
$$

In [ ]:
v = devm[devm["valid"]]
e_cfg = EXP["exposure"]
q_dev = ex.null_bootstrap_variation(daily, str(win["first_month"]), str(dev_last), reps=e_cfg["q_bootstrap_reps"],
                                    seed=EXP["seed"], nw_lags=e_cfg["newey_west_lags"], min_days=e_cfg["min_days_per_month"],
                                    block_days=e_cfg["q_bootstrap_block_days"], max_abs=e_cfg["max_abs_duration_flag"])
q_dev.update({"sample": f"{win['first_month']} to {dev_last}", "mean_D": float(v["D"].mean()),
              "p05_D": float(v["D"].quantile(0.05)), "p95_D": float(v["D"].quantile(0.95)), "median_se": float(v["se"].median())})
(xdir / "q1_variation_development.json").write_text(json.dumps(q_dev, indent=2))
print(f"Development months: {q_dev['months']} valid. Mean duration {q_dev['mean_D']:.2f} years, "
      f"5th-95th percentile {q_dev['p05_D']:.2f} to {q_dev['p95_D']:.2f} years. Median monthly se {q_dev['median_se']:.2f} years.")
print(f"Observed SD of monthly estimates {q_dev['sd_observed']:.2f} years. SD expected from noise alone {q_dev['sd_noise_under_null']:.2f} years.")
print(f"Robust between-month SD of true duration tau = {q_dev['tau_robust']:.2f} years. Cochran Q = {q_dev['Q']:.0f} on {q_dev['df']} df: "
      f"bootstrap p = {q_dev['p_bootstrap']:.3g} (naive chi-square p = {q_dev['p_chi2_naive']:.2g}, not used).")

In [ ]:
d = devm.copy(); d["date"] = d["decision_date"]
fig, ax = plt.subplots(figsize=(10, 4))
vv = d[d["valid"]]
ax.errorbar(vv["date"], vv["D"], yerr=vv["se"], fmt="o", ms=3, color="#2a78d6", ecolor="#9ec5f4", lw=0.8, label="realized duration, +/- 1 se")
hl_show = EXP["exposure"]["ewma_halflife_grid_days"][min(2, len(EXP["exposure"]["ewma_halflife_grid_days"]) - 1)]
ax.plot(d["date"], d[f"U_{hl_show}"], color="#eb6834", lw=1.6, label=f"EWMA duration, {hl_show}-day half-life")
ax.axhline(q_dev["mean_D"], color=pt.INK_2, lw=0.8, ls=":", label="development mean")
pt.shade_episodes(ax, [e for e in EXP["episodes"] if e["period"] == "development"])
ax.set_ylabel("years"); ax.set_title("Development period: monthly empirical duration of MBB"); ax.legend(loc="upper left", fontsize=8.5)
pt.save_figure(fig, "05_realized_duration_development", paths, data=d[["D", "se", f"U_{hl_show}"]], mode=MODE)
plt.show()

## 4. Duration and refinancing in development data (descriptive)

Does next month's duration line up with this month's share of borrowers in the money? The fit below is the same weighted regression the economic model uses, run once on development months. It describes the mechanism. It is **not** evidence of forecasting value, which only the walk-forward test in notebook 08 can provide. Hollow points rely on the retrospective proxy.

In [ ]:
from mbshedge.models import _wls
pairs = pd.DataFrame({"g": monthly["g"].shift(1), "D_next": monthly["D"], "se_next": monthly["se"],
                      "valid": monthly["valid"], "src": monthly["incentive_source"].shift(1)}).loc[win["first_month"] + 1:dev_last]
pairs = pairs[pairs["valid"] & pairs["g"].notna()]
c0, c1, tau2 = _wls(pairs["g"].to_numpy(), pairs["D_next"].to_numpy(), pairs["se_next"].to_numpy())
rel = {"c0_locked_in_duration": c0, "c1": c1, "duration_if_all_in_money": c0 + c1, "tau2": tau2, "n": int(len(pairs)),
       "share_proxy_months": float((pairs["src"] == "proxy_retrospective").mean()),
       "corr": float(pairs[["g", "D_next"]].corr().iloc[0, 1])}
(xdir / "state_relation_development.json").write_text(json.dumps(rel, indent=2))
fig, ax = plt.subplots(figsize=(6.5, 4.2))
nm = pairs["src"] == "nmdb"
ax.scatter(pairs.loc[~nm, "g"], pairs.loc[~nm, "D_next"], s=22, facecolors="none", edgecolors="#2a78d6", label="proxy months (retrospective)")
ax.scatter(pairs.loc[nm, "g"], pairs.loc[nm, "D_next"], s=22, color="#2a78d6", label="NMDB months")
xs = np.linspace(0, 1, 50); ax.plot(xs, c0 + c1 * xs, color=pt.INK, lw=1.4, label="weighted fit")
ax.set_xlabel("share of borrowers in the money at month-end"); ax.set_ylabel("next month's realized duration (years)")
ax.set_title("Development months: duration vs refinancing state"); ax.legend(fontsize=8)
pt.save_figure(fig, "05_duration_vs_state_development", paths, data=pairs, mode=MODE)
plt.show()
print(f"Development fit: locked-in duration c0 = {c0:.2f} years, all-in-the-money duration c0 + c1 = {c0 + c1:.2f} years, "
      f"correlation {rel['corr']:+.2f} over {rel['n']} months ({rel['share_proxy_months']:.0%} on the retrospective proxy).")

## 5. Timing check: does the 3:30 pm yield fix leak into the next day? (development months)

The Treasury yields behind DGS10 are fixed around 3:30 pm ET, while MBB's price is its 4 pm close. A rate move between 3:30 and 4:00 pm therefore enters today's ETF return but tomorrow's yield change. Adding the next day's yield change to each month's regression,

$$
r_t = a + b_{\text{lead}}\,\Delta y_{t+1} + b_0\,\Delta y_t + b_1\,\Delta y_{t-1} + e_t ,
$$

measures how much of the response the main specification misses. The next day's yield change is not known at the close, so this is an after-the-fact measurement. The main specification stays as declared. Every hedge in the project (static, updating, economic, hybrid) is built on the same estimator, so a missing lead would understate all of their durations by a similar share. That affects the level of every hedge more than the comparisons between them.

In [ ]:
ll = ex.lead_lag_diagnostic(daily, str(win["first_month"]), str(dev_last), min_days=EXP["exposure"]["min_days_per_month"],
                            nw_lags=EXP["exposure"]["newey_west_lags"])
ll_out = {k: v for k, v in ll.items() if k != "table"}
(xdir / "lead_lag_development.json").write_text(json.dumps(ll_out, indent=2))
if ll["months"]:
    ll["table"].to_csv(xdir / "lead_lag_development_months.csv")
    print(f"{ll['months']} development months. Mean lead effect {ll['mean_lead_years']:+.2f} years (se {ll['mean_lead_se']:.2f}), "
          f"mean lag effect {ll['mean_lag_years']:+.2f} years (se {ll['mean_lag_se']:.2f}).")
    print(f"Adding the lead changes the average monthly duration by {ll['mean_change_when_lead_added']:+.2f} years "
          f"(se {ll['mean_change_se']:.2f}), {ll['change_share_of_duration']:+.1%} of the lag-only average of {ll['mean_duration_lag_only']:.2f} years.")
    if abs(ll["mean_lead_years"]) > 2 * ll["mean_lead_se"]:
        direction = "understated" if ll["mean_change_when_lead_added"] > 0 else "overstated"
        print(f"The lead effect is statistically distinguishable from zero. Report it as a limitation: hedge ratios in this project "
              f"may be {direction} by about {abs(ll['change_share_of_duration']):.0%}, for every strategy alike.")
    else:
        print("The lead effect is not distinguishable from zero in development data.")